In [20]:
import kagglehub
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

path = kagglehub.dataset_download("mirichoi0218/insurance")

print("Path to dataset files:", path)

#read insureance dataset
insurance = pd.read_csv("C:/Users/LENOVO/.cache/kagglehub/datasets/mirichoi0218/insurance/versions/1/insurance.csv")

#check out insurance dataset
insurance.head(10)

Path to dataset files: C:\Users\LENOVO\.cache\kagglehub\datasets\mirichoi0218\insurance\versions\1


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
5,31,female,25.740,0,no,southeast,3756.62160
6,46,female,33.440,1,no,southeast,8240.58960
7,37,female,27.740,3,no,northwest,7281.50560
8,37,male,29.830,2,no,northeast,6406.41070
9,60,female,25.840,0,no,northwest,28923.13692


In [22]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# Create column transformer (this will help us normalize/preprocess our data)
ct = make_column_transformer(
    (MinMaxScaler(), ["age", "bmi", "children"]), # get all values between 0 and 1
    (OneHotEncoder(handle_unknown="ignore"), ["sex", "smoker", "region"])
)

# Create X & y
X = insurance.drop("charges", axis=1)
y = insurance["charges"]

# Build our train and test sets (use random state to ensure same split as before)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit column transformer on the training data only (doing so on test data would result in data leakage)
ct.fit(X_train)

# Transform training and test data with normalization (MinMaxScalar) and one hot encoding (OneHotEncoder)
X_train_scaled = ct.transform(X_train)
X_test_scaled = ct.transform(X_test)

# Reshape y because scalers expect 2D arrays
y_train = y_train.values.reshape(-1, 1)
y_test = y_test.values.reshape(-1, 1)
y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

In [24]:
y_train.shape

(1070, 1)

In [26]:
# Notice the normalized/one-hot encoded shape is larger because of the extra columns
X_train_scaled.shape, X_train.shape

((1070, 11), (1070, 6))

In [28]:
import tensorflow as tf
# Clear previous session (optional but clean for notebook environments)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1024, activation='relu'),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])
# Compile the model
# - Loss: Mean Absolute Error (MAE), ideal for regression problems
# - Optimizer: Adam (adaptive moment estimation), a smart choice for faster and more stable convergence
#   It adjusts the learning rate dynamically per parameter
model.compile(
    loss=tf.keras.losses.mae,
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001), # high LR for faster learning on small dataset
    metrics=['mae']
)


callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=20)
model.fit(X_train_scaled, y_train, epochs=300, verbose=1, callbacks=[callback])

Epoch 1/300
34/34 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - loss: 13293.2041 - mae: 13293.2041
Epoch 2/300
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 13112.5918 - mae: 13112.5918
Epoch 3/300
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 13354.4893 - mae: 13354.4893
Epoch 4/300
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 12847.6572 - mae: 12847.6572
Epoch 5/300
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 12768.6504 - mae: 12768.6504
Epoch 6/300
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 10494.8545 - mae: 10494.8545
Epoch 7/300
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 8331.5371 - mae: 8331.5371
Epoch 8/300
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 7827.6094 - mae: 7827.6094
Epoch 9/300
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 7729.8940 - mae: 7729.8940
Epoch 10/300
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 7392.9238 - mae: 7392.9238
Epoch 11/300
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 7186.1870 - mae: 7186.1870
Epoch 12/300
34/34 ━━━━━━━━━━

In [29]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                      │ (None, 1024)                │          12,288 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 512)                 │         524,800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,128,901 (8.12 MB)

 Trainable params: 709,633 (2.71 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,419,268 (5.41 MB)

In [30]:
import matplotlib.pyplot as plt

# Extract loss values
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(len(loss))

# Plot the loss curves
plt.figure(figsize=(8, 6))
plt.plot(epochs, loss, label='Training Loss')
plt.plot(epochs, val_loss, label='Validation Loss')
plt.title("Training vs Validation Loss Over Epochs")
plt.xlabel("Epochs")
plt.ylabel("MAE Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

NameError: name 'history' is not defined

In [ ]:
# Evaluate on test data
model.evaluate(X_test_scaled, y_test)

In [ ]:
from sklearn.metrics import mean_absolute_error

# Predict and inverse scale
y_pred_scaled = model.predict(X_test_scaled)
y_pred = y_scaler.inverse_transform(y_pred_scaled)
y_actual = y_scaler.inverse_transform(y_test_scaled)

# Real-world MAE in ₱
mae_peso = mean_absolute_error(y_actual, y_pred_scaled)
print(f"MAE in pesos: ₱{mae_peso:,.2f}")

In [ ]:
import numpy as np
import pandas as pd

# New sample for prediction
new_data = {
    "age": [19],
    "sex": ["female"],
    "bmi": [27.900],
    "children": [0],
    "smoker": ["yes"],
    "region": ["southwest"]
}

new_df = pd.DataFrame(new_data)

# Step 1: Preprocess the input features
new_df_transformed = ct.transform(new_df)

# Step 2: Predict the scaled charges
y_pred_scaled = model.predict(new_df_transformed)

# Step 3: Inverse scale to get prediction in original ₱ scale
y_pred = y_scaler.inverse_transform(y_pred_scaled)

# Display final prediction
print(f"Predicted Medical Charges: ₱{y_pred_scaled[0][0]:,.2f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

# Step 1: Predict on test set
y_pred_scaled = model.predict(X_test_normal)

# Step 2: Inverse scale predictions and actual values
y_pred = y_scaler.inverse_transform(y_pred_scaled)
y_actual = y_scaler.inverse_transform(y_test_scaled)

# Step 3: Plot Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_actual, y_pred_scaled, alpha=0.6, edgecolor='k')
plt.plot([y_actual.min(), y_actual.max()],
         [y_actual.min(), y_actual.max()],
         'r--', linewidth=2)  # Perfect prediction line

plt.xlabel("Actual Medical Charges (₱)")
plt.ylabel("Predicted Medical Charges (₱)")
plt.title("Actual vs Predicted Medical Charges")
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 4: Print MAE
mae_peso = mean_absolute_error(y_actual, y_pred)
print(f"Mean Absolute Error in pesos: ₱{mae_peso:,.2f}")

In [ ]:
# Predict charges for test data
y_preds = model.predict(X_test_normal)

In [ ]:
# Plot the model trained for 200 total epochs loss curves
pd.DataFrame(history.history).plot()
plt.ylabel("loss")
plt.xlabel("epochs"); # note: epochs will only show 100 since we overrid the history variable